# Создание размеченного датасета

**Подход:** Используем DuckDB для работы с большими данными без загрузки в память

**Результат:** `data/raw/chatbots_dataset_labelled.parquet` — полный датасет с двумя дополнительными колонками:
- `message_label` — разметка конкретных сообщений (illegal/legal)
- `from_illegal_account` — бинарный флаг нелегальных аккаунтов (1/0)


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

## 1. Подготовка путей

In [38]:
# Пути к файлам
data_dir = Path('../..') / 'data'
input_parquet = data_dir / 'raw' / 'chatbots_dataset.parquet'
labeled_csv = data_dir / 'processed' / 'full_razmetka.csv'
output_parquet = data_dir / 'raw' / 'chatbots_dataset_labelled.parquet'

# Проверяем существование файлов
if not input_parquet.exists():
    raise FileNotFoundError(f"Файл не найден: {input_parquet}")
if not labeled_csv.exists():
    raise FileNotFoundError(f"Файл не найден: {labeled_csv}")

print(f"Исходный датасет: {input_parquet}")
print(f"Размер: {input_parquet.stat().st_size / (1024**2):.2f} МБ")
print(f"Разметка: {labeled_csv}")
print(f"Размер: {labeled_csv.stat().st_size / (1024**2):.2f} МБ")

Исходный датасет: ..\..\data\raw\chatbots_dataset.parquet
Размер: 279.64 МБ
Разметка: ..\..\data\processed\full_razmetka.csv
Размер: 0.31 МБ


## 2. Загрузка и подготовка разметки

In [39]:
# Загружаем разметку (она небольшая, можно в pandas)
try:
    df_labeled = pd.read_csv(labeled_csv, sep=';', encoding='utf-8-sig')
    print(f"Загружена разметка с разделителем ';'")
except:
    df_labeled = pd.read_csv(labeled_csv, encoding='utf-8-sig')
    print(f"Загружена разметка с разделителем ','")

print(f"Всего строк: {len(df_labeled)}")
print(f"\nРаспределение меток:")
print(df_labeled['label'].value_counts(dropna=False))

Загружена разметка с разделителем ';'
Всего строк: 872

Распределение меток:
label
legal      508
illegal    363
NaN          1
Name: count, dtype: int64


In [40]:
# Фильтруем только валидные метки
df_labeled = df_labeled[
    df_labeled['label'].notna() & 
    (df_labeled['label'] != '') & 
    df_labeled['label'].isin(['illegal', 'legal'])
].copy()

print(f"Валидно размеченных: {len(df_labeled)}")
print(f"  illegal: {(df_labeled['label'] == 'illegal').sum()}")
print(f"  legal: {(df_labeled['label'] == 'legal').sum()}")

Валидно размеченных: 871
  illegal: 363
  legal: 508


In [41]:
# Создаем уникальный ключ для джойна
df_labeled['message_key'] = (
    df_labeled['account_id'].astype(str) + '|||' + 
    df_labeled['session_id'].astype(str) + '|||' + 
    df_labeled['question'].astype(str)
)

# Оставляем только нужные колонки
df_labeled_small = df_labeled[['message_key', 'label']].copy()
df_labeled_small = df_labeled_small.rename(columns={'label': 'message_label'})

print(f"\nПодготовлено {len(df_labeled_small)} записей для джойна")


Подготовлено 871 записей для джойна


## 3. Сохранение разметки во временный parquet

DuckDB работает быстрее с parquet, чем с CSV.

In [42]:
# Сохраняем разметку во временный parquet
temp_labels = data_dir / 'processed' / 'temp_labels.parquet'
df_labeled_small.to_parquet(temp_labels, engine='pyarrow', index=False)

print(f"Разметка сохранена во временный файл: {temp_labels}")
print(f"Размер: {temp_labels.stat().st_size / (1024):.2f} КБ")

Разметка сохранена во временный файл: ..\..\data\processed\temp_labels.parquet
Размер: 26.36 КБ


## 4. Обработка через DuckDB

Создаем новый датасет с двумя дополнительными колонками.

In [43]:
# Нелегальные аккаунты
illegal_accounts = ['7012008969', '4381336587', '509107573', '322189240', '8412110593']

print(f"Нелегальные аккаунты ({len(illegal_accounts)}):")
for acc in illegal_accounts:
    print(f"  - {acc}")

Нелегальные аккаунты (5):
  - 7012008969
  - 4381336587
  - 509107573
  - 322189240
  - 8412110593


In [44]:
# Подключаемся к DuckDB
con = duckdb.connect(database=':memory:')

# Формируем SQL-запрос для создания размеченного датасета
accounts_list = "', '".join(illegal_accounts)

query = f"""
SELECT 
    main.*,
    -- Добавляем флаг нелегального аккаунта
    CASE 
        WHEN main.account_id IN ('{accounts_list}') THEN 1 
        ELSE 0 
    END AS from_illegal_account,
    -- Добавляем разметку сообщений (все неразмеченные = legal)
    COALESCE(labels.message_label, 'legal') AS message_label
FROM '{input_parquet}' AS main
LEFT JOIN '{temp_labels}' AS labels
    ON (main.account_id || '|||' || main.session_id || '|||' || main.question) = labels.message_key
"""

print("SQL-запрос подготовлен")

SQL-запрос подготовлен


In [45]:
# Выполняем запрос и сохраняем результат напрямую в parquet
print(f"\nВыполнение запроса и сохранение в {output_parquet}...")

con.execute(f"""
COPY (
    {query}
) TO '{output_parquet}' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

print(f"\nДатасет сохранен!")
print(f"Файл: {output_parquet}")
print(f"Размер: {output_parquet.stat().st_size / (1024**2):.2f} МБ")


Выполнение запроса и сохранение в ..\..\data\raw\chatbots_dataset_labelled.parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Датасет сохранен!
Файл: ..\..\data\raw\chatbots_dataset_labelled.parquet
Размер: 624.11 МБ


## 5. Проверка результата

In [46]:
# Статистика по новому датасету
stats_query = f"""
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN message_label = 'illegal' THEN 1 ELSE 0 END) as illegal_count,
    SUM(CASE WHEN message_label = 'legal' THEN 1 ELSE 0 END) as legal_count,
    SUM(CASE WHEN from_illegal_account = 1 THEN 1 ELSE 0 END) as from_illegal_account_count
FROM '{output_parquet}';
"""

stats = con.execute(stats_query).df()

print("Статистика итогового датасета:")
print(f"  Всего строк: {stats['total_rows'][0]:,}")
print(f"\nРаспределение message_label:")
print(f"  illegal: {stats['illegal_count'][0]:,} ({stats['illegal_count'][0] / stats['total_rows'][0] * 100:.2f}%)")
print(f"  legal: {stats['legal_count'][0]:,} ({stats['legal_count'][0] / stats['total_rows'][0] * 100:.2f}%)")
print(f"\nРаспределение from_illegal_account:")
print(f"  from_illegal_account=1: {stats['from_illegal_account_count'][0]:,} ({stats['from_illegal_account_count'][0] / stats['total_rows'][0] * 100:.2f}%)")
print(f"  from_illegal_account=0: {stats['total_rows'][0] - stats['from_illegal_account_count'][0]:,} ({(1 - stats['from_illegal_account_count'][0] / stats['total_rows'][0]) * 100:.2f}%)")

Статистика итогового датасета:
  Всего строк: 7,816,579

Распределение message_label:
  illegal: 463.0 (0.01%)
  legal: 7,816,116.0 (99.99%)

Распределение from_illegal_account:
  from_illegal_account=1: 1,431.0 (0.02%)
  from_illegal_account=0: 7,815,148.0 (99.98%)


In [51]:
# Примеры записей
examples_query = f"""
  SELECT * FROM (
      -- Примеры illegal сообщений
      SELECT 'illegal' as type, account_id, question, answer, message_label, from_illegal_account
      FROM '{output_parquet}'
      WHERE message_label = 'illegal'
      LIMIT 5
  ) 
  UNION ALL
  SELECT * FROM (
      -- Примеры legal из нелегальных аккаунтов
      SELECT 'legal_from_illegal' as type, account_id, question, answer, message_label, from_illegal_account
      FROM '{output_parquet}'
      WHERE message_label = 'legal' AND from_illegal_account = 1
      LIMIT 5
  )
  UNION ALL
  SELECT * FROM (                                                                                                                            
      -- Примеры legal из легальных аккаунтов
      SELECT 'legal_from_legal' as type, account_id, question, answer, message_label, from_illegal_account
      FROM '{output_parquet}'
      WHERE message_label = 'legal' AND from_illegal_account = 0
      LIMIT 5
  )
  """

examples = con.execute(examples_query).df()

print("\nПримеры записей (question + answer):")
for type_name in ['illegal', 'legal_from_illegal', 'legal_from_legal']:
    print(f"\n{type_name}:")
    display(examples[examples['type'] == type_name][['account_id', 'question', 'answer', 'message_label', 'from_illegal_account']]) 


Примеры записей (question + answer):

illegal:


,account_id,question,answer,message_label,from_illegal_account
0,322189240,/start,Добро пожаловать в бота автопродаж магазина @C...,illegal,1
1,322189240,/start,Добро пожаловать в бота автопродаж магазина @C...,illegal,1
2,322189240,/newNode_3;Альфа-ПВП VHQ+ Крисы 0.25 гр.,Избран продукт: Альфа-ПВП VHQ+ Крисы 0.25 гр.\...,illegal,1
3,322189240,/newNode_9;Жовтневый,За последние сутки Вы уже отменили 1 заказов. ...,illegal,1
4,322189240,/newNode_15;Новый EasyPay,Ваш заказ № 7085:\nГород: Мариуполь.\nРайон: Ж...,illegal,1



legal_from_illegal:


,account_id,question,answer,message_label,from_illegal_account
5,322189240,,,legal,1
6,322189240,/newNode_21;Отменить заказ,Заказ № 7085 будет отменен. Ты уверен?!,legal,1
7,322189240,,,legal,1
8,322189240,,,legal,1
9,322189240,,,legal,1



legal_from_legal:


,account_id,question,answer,message_label,from_illegal_account
10,591417391,Нет,"Пожалуйста, напишите Ваш вопрос в краткой форм...",legal,0
11,591417391,Нет,"Уточните, пожалуйста, Вам уже одобрена заявка ...",legal,0
12,138458287,🛫 Вылет из DME,Выберите или укажите дату.,legal,0
13,183263525,⚡️ работа по всей украине!\n💰 20 000+ грн/день...,None,legal,0
14,183263525,⚡️ работа по всей украине!\n💰 20 000+ грн/день...,None,legal,0


## 6. Создание sample-датасета

Небольшой датасет для быстрого тестирования.

In [53]:
# Создаем sample: все illegal + legal из нелегальных аккаунтов + 10k случайных legal
sample_output = data_dir / 'raw' / 'chatbots_dataset_sample.parquet'

sample_query = f"""
COPY (
    -- Все illegal
    (SELECT * FROM '{output_parquet}' WHERE message_label = 'illegal')
    UNION ALL
    -- Все legal из нелегальных аккаунтов
    (SELECT * FROM '{output_parquet}' WHERE message_label = 'legal' AND from_illegal_account = 1)
    UNION ALL
    -- Случайные 10k legal из легальных аккаунтов
    (SELECT * FROM '{output_parquet}' 
     WHERE message_label = 'legal' AND from_illegal_account = 0 
     ORDER BY RANDOM() 
     LIMIT 10000)
) TO '{sample_output}' (FORMAT PARQUET, COMPRESSION SNAPPY)
"""

print("Создание sample-датасета...")
con.execute(sample_query)

print(f"\nSample-датасет создан!")
print(f"Файл: {sample_output}")
print(f"Размер: {sample_output.stat().st_size / (1024**2):.2f} МБ")

Создание sample-датасета...

Sample-датасет создан!
Файл: ..\..\data\raw\chatbots_dataset_sample.parquet
Размер: 1.27 МБ


In [15]:
# Статистика sample
sample_stats_query = f"""
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN message_label = 'illegal' THEN 1 ELSE 0 END) as illegal_count,
    SUM(CASE WHEN message_label = 'legal' THEN 1 ELSE 0 END) as legal_count
FROM '{sample_output}';
"""

sample_stats = con.execute(sample_stats_query).df()

print("Статистика sample:")
print(f"  Всего строк: {sample_stats['total_rows'][0]:,}")
print(f"  illegal: {sample_stats['illegal_count'][0]:,}")
print(f"  legal: {sample_stats['legal_count'][0]:,}")

Статистика sample:
  Всего строк: 11,431
  illegal: 463.0
  legal: 10,968.0


## 7. Очистка

In [54]:
# Удаляем временный файл
if temp_labels.exists():
    temp_labels.unlink()
    print(f"✓ Удален временный файл: {temp_labels}")

# Закрываем подключение
con.close()
print("✓ Подключение закрыто")

✓ Удален временный файл: ..\..\data\processed\temp_labels.parquet
✓ Подключение закрыто


## 8. Итоговая сводка

In [57]:
print(f"\nПОЛНЫЙ ДАТАСЕТ")
print(f"Файл: data/raw/chatbots_dataset.parquet")
print(f"Размер: {output_parquet.stat().st_size / (1024**2):.2f} МБ")

print(f"\nSAMPLE-ДАТАСЕТ (для быстрого тестирования)")
print(f"Файл: data/raw/chatbots_dataset_sample.parquet")
print(f"Размер: {sample_output.stat().st_size / (1024**2):.2f} МБ")

print(f"\nСтруктура датасетов:")
print(f"Все исходные колонки (account_id, session_id, question, text_answer, и т.д.)")
print(f"   + 2 новые колонки:")
print(f"   • message_label — illegal/legal")
print(f"     Все сообщения НЕ из ручной разметки → автоматически 'legal'")
print(f"   • from_illegal_account — 1/0")
print(f"     1 для аккаунтов: {', '.join(illegal_accounts)}")


ПОЛНЫЙ ДАТАСЕТ
Файл: data/raw/chatbots_dataset.parquet
Размер: 624.11 МБ

SAMPLE-ДАТАСЕТ (для быстрого тестирования)
Файл: data/raw/chatbots_dataset_sample.parquet
Размер: 1.27 МБ

Структура датасетов:
Все исходные колонки (account_id, session_id, question, text_answer, и т.д.)
   + 2 новые колонки:
   • message_label — illegal/legal
     Все сообщения НЕ из ручной разметки → автоматически 'legal'
   • from_illegal_account — 1/0
     1 для аккаунтов: 7012008969, 4381336587, 509107573, 322189240, 8412110593
